In [2]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

import pandas as pd

file_path = '/content/drive/MyDrive/CareLoop_NoShow_Guard/appointments_clean.csv'

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (990, 16)

First 5 rows:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [4]:

feature_cols = [
    'age',
    'distance_km',
    'fee_inr',
    'lead_days',
    'prior_no_shows',
    'reminder_sent',
    'booking_channel',
    'visit_type'
]

X = df[feature_cols]
y = df['no_show']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())


Features shape: (990, 8)
Target shape: (990,)

Target distribution:
no_show
0    772
1    218
Name: count, dtype: int64


In [5]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Training features: (792, 8)
Testing features: (198, 8)
Training target: (792,)
Testing target: (198,)


In [6]:

categorical_cols = [
    'reminder_sent',
    'booking_channel',
    'visit_type'
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(X_train[col].value_counts())



reminder_sent:
reminder_sent
Yes    452
No     340
Name: count, dtype: int64

booking_channel:
booking_channel
App        303
Phone      288
Walk-In    201
Name: count, dtype: int64

visit_type:
visit_type
Consultation    463
Follow-up       329
Name: count, dtype: int64


In [7]:

from sklearn.preprocessing import OneHotEncoder

# Create the encoder
encoder = OneHotEncoder(
    drop='first',
    handle_unknown='ignore',
    sparse_output=False
)

# Learn categories from training data
X_train_encoded = encoder.fit_transform(
    X_train[categorical_cols]
)

# Apply the same encoding to testing data
X_test_encoded = encoder.transform(
    X_test[categorical_cols]
)

# Display the encoded column names
print("Encoded columns:")
print(encoder.get_feature_names_out(categorical_cols))

print("\nTraining encoded shape:", X_train_encoded.shape)
print("Testing encoded shape:", X_test_encoded.shape)


Encoded columns:
['reminder_sent_Yes' 'booking_channel_Phone' 'booking_channel_Walk-In'
 'visit_type_Follow-up']

Training encoded shape: (792, 4)
Testing encoded shape: (198, 4)


In [8]:

# Select numerical columns
numerical_cols = [
    'age',
    'distance_km',
    'fee_inr',
    'lead_days',
    'prior_no_shows'
]

# Convert encoded arrays into DataFrames
train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

# Combine numerical and encoded features
X_train_final = pd.concat(
    [X_train[numerical_cols], train_encoded_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test[numerical_cols], test_encoded_df],
    axis=1
)

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

display(X_train_final.head())


Final training shape: (792, 9)
Final testing shape: (198, 9)


,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent_Yes,booking_channel_Phone,booking_channel_Walk-In,visit_type_Follow-up
753,51,11.8,800,15,0,0.0,1.0,0.0,0.0
215,76,18.7,300,2,0,0.0,1.0,0.0,1.0
294,58,7.0,800,14,0,0.0,1.0,0.0,0.0
49,41,8.2,600,8,0,0.0,0.0,1.0,0.0
720,57,25.6,700,4,0,0.0,0.0,1.0,0.0


In [9]:

from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Make copies to preserve original features
X_train_scaled = X_train_final.copy()
X_test_scaled = X_test_final.copy()

# Learn mean and standard deviation from training data
X_train_scaled[numerical_cols] = scaler.fit_transform(
    X_train_final[numerical_cols]
)

# Apply the same scaling to testing data
X_test_scaled[numerical_cols] = scaler.transform(
    X_test_final[numerical_cols]
)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nMean of scaled numerical features:")
print(X_train_scaled[numerical_cols].mean().round(2))

print("\nStandard deviation:")
print(X_train_scaled[numerical_cols].std(ddof=0).round(2))


Scaled training shape: (792, 9)
Scaled testing shape: (198, 9)

Mean of scaled numerical features:
age               0.0
distance_km       0.0
fee_inr          -0.0
lead_days         0.0
prior_no_shows   -0.0
dtype: float64

Standard deviation:
age               1.0
distance_km       1.0
fee_inr           1.0
lead_days         1.0
prior_no_shows    1.0
dtype: float64


In [10]:

from sklearn.linear_model import LogisticRegression

# Create the model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using training data
lr_model.fit(X_train_scaled, y_train)

# Predict outcomes for unseen testing data
y_pred_lr = lr_model.predict(X_test_scaled)

# Predict probabilities of no-show
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Model trained successfully!")

print("\nFirst 10 predictions:")
print(y_pred_lr[:10])

print("\nFirst 10 no-show probabilities:")
print(y_prob_lr[:10].round(3))


Model trained successfully!

First 10 predictions:
[0 0 1 0 0 0 0 0 0 0]

First 10 no-show probabilities:
[0.022 0.148 0.511 0.188 0.171 0.319 0.342 0.361 0.107 0.176]


In [11]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred_lr)
precision = precision_score(y_test, y_pred_lr, zero_division=0)
recall = recall_score(y_test, y_pred_lr, zero_division=0)
f1 = f1_score(y_test, y_pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob_lr)

print("LOGISTIC REGRESSION RESULTS")
print("---------------------------")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))


LOGISTIC REGRESSION RESULTS
---------------------------
Accuracy:  0.788
Precision: 0.630
Recall:    0.347
F1-score:  0.447
ROC-AUC:   0.789

Confusion Matrix:
[[139  10]
 [ 32  17]]


In [12]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Create the Decision Tree model
dt_model = DecisionTreeClassifier(
    random_state=42
)

# Train using training data
dt_model.fit(X_train_scaled, y_train)

# Make predictions on test data
y_pred_dt = dt_model.predict(X_test_scaled)
y_prob_dt = dt_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
print("DECISION TREE RESULTS")
print("---------------------")
print("Training Accuracy:",
      round(dt_model.score(X_train_scaled, y_train), 3))

print("Testing Accuracy:",
      round(accuracy_score(y_test, y_pred_dt), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, y_prob_dt), 3))

print("Tree Depth:", dt_model.get_depth())


DECISION TREE RESULTS
---------------------
Training Accuracy: 1.0
Testing Accuracy: 0.697
ROC-AUC: 0.607
Tree Depth: 15


In [13]:

# Create a simpler Decision Tree
dt_pruned = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the pruned model
dt_pruned.fit(X_train_scaled, y_train)

# Predict on testing data
y_pred_dt_pruned = dt_pruned.predict(X_test_scaled)
y_prob_dt_pruned = dt_pruned.predict_proba(
    X_test_scaled
)[:, 1]

# Compare training and testing performance
print("PRUNED DECISION TREE RESULTS")
print("----------------------------")

print("Training Accuracy:",
      round(dt_pruned.score(X_train_scaled, y_train), 3))

print("Testing Accuracy:",
      round(accuracy_score(y_test, y_pred_dt_pruned), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, y_prob_dt_pruned), 3))

print("Tree Depth:", dt_pruned.get_depth())


PRUNED DECISION TREE RESULTS
----------------------------
Training Accuracy: 0.822
Testing Accuracy: 0.778
ROC-AUC: 0.7
Tree Depth: 3


In [14]:

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 trees
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate performance
print("RANDOM FOREST RESULTS")
print("---------------------")

print("Training Accuracy:",
      round(rf_model.score(X_train_scaled, y_train), 3))

print("Testing Accuracy:",
      round(accuracy_score(y_test, y_pred_rf), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, y_prob_rf), 3))


RANDOM FOREST RESULTS
---------------------
Training Accuracy: 1.0
Testing Accuracy: 0.768
ROC-AUC: 0.699


In [15]:

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Scale numerical features; leave encoded features unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
)

# Create a pipeline
lr_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Create 5 stratified folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Calculate ROC-AUC on each validation fold
cv_scores_lr = cross_val_score(
    lr_pipeline,
    X_train_final,
    y_train,
    cv=cv,
    scoring='roc_auc'
)

print("ROC-AUC for each fold:")
print(cv_scores_lr.round(3))

print("\nMean CV ROC-AUC:",
      round(cv_scores_lr.mean(), 3))

print("Standard Deviation:",
      round(cv_scores_lr.std(), 3))


ROC-AUC for each fold:
[0.792 0.762 0.788 0.782 0.761]

Mean CV ROC-AUC: 0.777
Standard Deviation: 0.013


In [16]:

from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Models to compare
models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# Cross-validation for each model
for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_final,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    print(f"\n{name}")
    print("Fold ROC-AUC:", scores.round(3))
    print("Mean ROC-AUC:", round(scores.mean(), 3))
    print("Std Deviation:", round(scores.std(), 3))



Decision Tree
Fold ROC-AUC: [0.661 0.676 0.641 0.668 0.66 ]
Mean ROC-AUC: 0.661
Std Deviation: 0.012

Random Forest
Fold ROC-AUC: [0.708 0.766 0.755 0.712 0.807]
Mean ROC-AUC: 0.749
Std Deviation: 0.037


In [17]:

from sklearn.model_selection import GridSearchCV

# Create the Random Forest model
rf_tuning = RandomForestClassifier(
    random_state=42
)

# Hyperparameter combinations to test
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8]
}

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_tuning,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

# Search for the best combination
grid_search.fit(X_train_final, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Mean CV ROC-AUC:")
print(round(grid_search.best_score_, 3))


Best Parameters:
{'max_depth': 8, 'n_estimators': 200}

Best Mean CV ROC-AUC:
0.767


In [18]:

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Generate out-of-fold probabilities using training data
train_probabilities = cross_val_predict(
    lr_pipeline,
    X_train_final,
    y_train,
    cv=cv,
    method='predict_proba'
)[:, 1]

# Compare different classification thresholds
thresholds = [0.30, 0.35, 0.40, 0.50]

print("THRESHOLD COMPARISON")
print("--------------------")

for threshold in thresholds:

    predictions = (
        train_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_train, predictions, zero_division=0
    )

    recall = recall_score(
        y_train, predictions, zero_division=0
    )

    f1 = f1_score(
        y_train, predictions, zero_division=0
    )

    flagged = predictions.sum()

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f} | "
        f"Flagged: {flagged}"
    )


THRESHOLD COMPARISON
--------------------
Threshold: 0.30 | Precision: 0.471 | Recall: 0.533 | F1: 0.500 | Flagged: 191
Threshold: 0.35 | Precision: 0.527 | Recall: 0.467 | F1: 0.495 | Flagged: 150
Threshold: 0.40 | Precision: 0.547 | Recall: 0.414 | F1: 0.471 | Flagged: 128
Threshold: 0.50 | Precision: 0.609 | Recall: 0.249 | F1: 0.353 | Flagged: 69


In [19]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Train final Logistic Regression pipeline
lr_pipeline.fit(X_train_final, y_train)

# Predict no-show probabilities on test data
final_probabilities = lr_pipeline.predict_proba(
    X_test_final
)[:, 1]

# Apply threshold selected using training CV
final_threshold = 0.30

final_predictions = (
    final_probabilities >= final_threshold
).astype(int)

# Evaluate final model
print("FINAL LOGISTIC REGRESSION RESULTS")
print("---------------------------------")

print("Threshold:", final_threshold)

print("Accuracy:",
      round(accuracy_score(y_test, final_predictions), 3))

print("Precision:",
      round(precision_score(y_test, final_predictions,
                            zero_division=0), 3))

print("Recall:",
      round(recall_score(y_test, final_predictions,
                         zero_division=0), 3))

print("F1-score:",
      round(f1_score(y_test, final_predictions,
                     zero_division=0), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, final_probabilities), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nPatients flagged:",
      final_predictions.sum())


FINAL LOGISTIC REGRESSION RESULTS
---------------------------------
Threshold: 0.3
Accuracy: 0.793
Precision: 0.571
Recall: 0.653
F1-score: 0.61
ROC-AUC: 0.789

Confusion Matrix:
[[125  24]
 [ 17  32]]

Patients flagged: 56


In [20]:

import os
import pandas as pd

# Create results for the 198 test appointments
risk_scores = pd.DataFrame({
    'appointment_id': df.loc[X_test_final.index, 'appointment_id'].values,
    'no_show': y_test.values,
    'risk_prob': final_probabilities.round(4),
    'flag': final_predictions
})

# Save directly in the main project folder
output_path = (
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'test_risk_scores.csv'
)

risk_scores.to_csv(output_path, index=False)

print("File saved:", os.path.exists(output_path))
print("Dataset shape:", risk_scores.shape)
print("Total flagged:", risk_scores['flag'].sum())

print("\nFirst 5 records:")
display(risk_scores.head())


File saved: True
Dataset shape: (198, 4)
Total flagged: 56

First 5 records:


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.0222,0
1,A0336,0,0.1483,0
2,A0508,0,0.5109,1
3,A0314,0,0.1881,0
4,A0089,0,0.1706,0


In [21]:

import joblib
import os

# Save the complete pipeline:
# StandardScaler + Logistic Regression
model_path = (
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'noshow_model.joblib'
)

joblib.dump(lr_pipeline, model_path)

print("Model saved:", os.path.exists(model_path))

# Load the saved model to verify it works
loaded_model = joblib.load(model_path)

# Test the loaded model on one appointment
sample = X_test_final.iloc[[0]]

probability = loaded_model.predict_proba(sample)[0, 1]

print("Sample no-show probability:",
      round(probability, 3))

print("Sample risk flag:",
      int(probability >= 0.30))


Model saved: True
Sample no-show probability: 0.022
Sample risk flag: 0


In [22]:

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Training-set out-of-fold probabilities
oof_probs = cross_val_predict(
    lr_pipeline,
    X_train_final,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

results = []

for threshold in [0.50, 0.35]:
    predictions = (oof_probs >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": round(precision_score(y_train, predictions), 3),
        "Recall": round(recall_score(y_train, predictions), 3),
        "F1": round(f1_score(y_train, predictions), 3),
        "Flagged": int(predictions.sum())
    })

comparison = pd.DataFrame(results)

display(comparison)

best_threshold = comparison.loc[
    comparison["F1"].idxmax(), "Threshold"
]

print("Selected threshold:", best_threshold)


,Threshold,Precision,Recall,F1,Flagged
0,0.50,0.595,0.260,0.362,74
1,0.35,0.510,0.456,0.481,151


Selected threshold: 0.35


In [23]:

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
import pandas as pd

# Train final model using training data only
lr_pipeline.fit(X_train_final, y_train)

# Predict probabilities on held-out test data
test_probs = lr_pipeline.predict_proba(
    X_test_final
)[:, 1]

# Apply selected threshold
test_preds = (test_probs >= 0.35).astype(int)

print("FINAL TEST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, test_preds), 3))
print("Precision:", round(precision_score(y_test, test_preds), 3))
print("Recall:", round(recall_score(y_test, test_preds), 3))
print("F1-score:", round(f1_score(y_test, test_preds), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 3))
print("Appointments flagged:", int(test_preds.sum()))

# Update risk scores CSV
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test_final.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_preds
})

output_path = (
    "/content/drive/MyDrive/CareLoop_NoShow_Guard/"
    "test_risk_scores.csv"
)

risk_scores.to_csv(output_path, index=False)

print("Updated test_risk_scores.csv successfully!")


FINAL TEST RESULTS
Accuracy: 0.788
Precision: 0.571
Recall: 0.571
F1-score: 0.571
ROC-AUC: 0.789
Appointments flagged: 49
Updated test_risk_scores.csv successfully!
